In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    lit,
    lpad,
    concat_ws,
    to_date
)
from pyspark.sql.window import Window

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 3, Finished, Available, Finished, False)

In [2]:
t100 = spark.table("bronze_t100_domestic_segment")

dim_date = spark.table("dim_date")
dim_airline = spark.table("dim_airline")
dim_airport = spark.table("dim_airport")
dim_route = spark.table("dim_route")

print("T-100 Source Rows:", t100.count())
print("Date Dimension Rows:", dim_date.count())
print("Airline Dimension Rows:", dim_airline.count())
print("Airport Dimension Rows:", dim_airport.count())
print("Route Dimension Rows:", dim_route.count())

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 4, Finished, Available, Finished, False)

T-100 Source Rows: 36186
Date Dimension Rows: 31
Airline Dimension Rows: 155
Airport Dimension Rows: 6072
Route Dimension Rows: 11920


In [3]:
t100_std = (
    t100
    .select(
        col("DEPARTURES_SCHEDULED").alias("departures_scheduled"),
        col("PAYLOAD").alias("payload"),
        col("SEATS").alias("seats"),
        col("PASSENGERS").alias("passengers"),
        col("FREIGHT").alias("freight"),
        col("MAIL").alias("mail"),
        col("DISTANCE").alias("distance"),
        col("RAMP_TO_RAMP").alias("ramp_to_ramp"),
        col("AIR_TIME").alias("air_time"),

        col("AIRLINE_ID").cast("long").alias("airline_id"),
        upper(trim(col("UNIQUE_CARRIER"))).alias("unique_carrier"),

        upper(trim(col("ORIGIN"))).alias("origin"),
        upper(trim(col("DEST"))).alias("destination"),

        col("AIRCRAFT_TYPE").alias("aircraft_type"),

        col("YEAR").cast("int").alias("year"),
        col("MONTH").cast("int").alias("month")
    )
    .withColumn(
        "operation_date",
        to_date(
            concat_ws(
                "-",
                col("year"),
                lpad(col("month"), 2, "0"),
                lit("01")
            )
        )
    )
)

print("Standardized Rows:", t100_std.count())
t100_std.printSchema()

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 5, Finished, Available, Finished, False)

Standardized Rows: 36186
root
 |-- departures_scheduled: long (nullable = true)
 |-- payload: long (nullable = true)
 |-- seats: long (nullable = true)
 |-- passengers: long (nullable = true)
 |-- freight: long (nullable = true)
 |-- mail: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- ramp_to_ramp: long (nullable = true)
 |-- air_time: long (nullable = true)
 |-- airline_id: long (nullable = true)
 |-- unique_carrier: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- aircraft_type: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- operation_date: date (nullable = true)



In [4]:
date_lookup = (
    dim_date
    .select(
        "date_key",
        col("flight_date").alias("operation_date")
    )
)

airline_lookup = (
    dim_airline
    .select(
        "airline_key",
        "airline_id"
    )
)

origin_lookup = (
    dim_airport
    .select(
        col("airport_key").alias("origin_airport_key"),
        upper(trim(col("iata_code"))).alias("origin")
    )
)

destination_lookup = (
    dim_airport
    .select(
        col("airport_key").alias("destination_airport_key"),
        upper(trim(col("iata_code"))).alias("destination")
    )
)

route_lookup = (
    dim_route
    .select(
        "route_key",
        upper(trim(col("origin_airport"))).alias("route_origin"),
        upper(trim(col("destination_airport"))).alias("route_destination")
    )
)

print("Lookups created.")

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 6, Finished, Available, Finished, False)

Lookups created.


In [5]:
traffic_enriched = (
    t100_std.alias("t")

    .join(
        airline_lookup.alias("a"),
        col("t.airline_id") == col("a.airline_id"),
        "left"
    )

    .join(
        date_lookup.alias("d"),
        col("t.operation_date") == col("d.operation_date"),
        "left"
    )

    .join(
        origin_lookup.alias("o"),
        col("t.origin") == col("o.origin"),
        "left"
    )

    .join(
        destination_lookup.alias("da"),
        col("t.destination") == col("da.destination"),
        "left"
    )

    .join(
        route_lookup.alias("r"),
        (
            col("t.origin") == col("r.route_origin")
        )
        &
        (
            col("t.destination") == col("r.route_destination")
        ),
        "left"
    )

    .select(
        col("d.date_key"),
        col("a.airline_key"),
        col("o.origin_airport_key"),
        col("da.destination_airport_key"),
        col("r.route_key"),

        col("t.aircraft_type"),
        col("t.departures_scheduled"),
        col("t.payload"),
        col("t.seats"),
        col("t.passengers"),
        col("t.freight"),
        col("t.mail"),
        col("t.distance"),
        col("t.ramp_to_ramp"),
        col("t.air_time")
    )
)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 7, Finished, Available, Finished, False)

In [6]:
source_rows = t100_std.count()
joined_rows = traffic_enriched.count()

print("Source Rows:", source_rows)
print("Rows After Dimension Joins:", joined_rows)
print("Row Difference:", joined_rows - source_rows)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 8, Finished, Available, Finished, False)

Source Rows: 36186
Rows After Dimension Joins: 36186
Row Difference: 0


In [7]:
traffic_key_validation = traffic_enriched.agg(
    F.sum(
        F.when(col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_keys"),

    F.sum(
        F.when(col("airline_key").isNull(), 1).otherwise(0)
    ).alias("null_airline_keys"),

    F.sum(
        F.when(col("origin_airport_key").isNull(), 1).otherwise(0)
    ).alias("null_origin_airport_keys"),

    F.sum(
        F.when(col("destination_airport_key").isNull(), 1).otherwise(0)
    ).alias("null_destination_airport_keys"),

    F.sum(
        F.when(col("route_key").isNull(), 1).otherwise(0)
    ).alias("null_route_keys")
)

display(traffic_key_validation)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ca0243cd-66b3-4fd3-9684-a3766ffe2506)

In [8]:
traffic_valid = (
    traffic_enriched
    .filter(col("origin_airport_key").isNotNull())
    .filter(col("destination_airport_key").isNotNull())
)

rows_before_filter = traffic_enriched.count()
rows_after_filter = traffic_valid.count()

print("Rows Before Airport Coverage Filter:", rows_before_filter)
print("Rows After Airport Coverage Filter:", rows_after_filter)
print(
    "Rows Excluded For Airport Reference Coverage:",
    rows_before_filter - rows_after_filter
)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 10, Finished, Available, Finished, False)

Rows Before Airport Coverage Filter: 36186
Rows After Airport Coverage Filter: 34902
Rows Excluded For Airport Reference Coverage: 1284


In [9]:
print(
    "Null Airline Keys:",
    traffic_valid
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Null Route Keys:",
    traffic_valid
    .filter(col("route_key").isNull())
    .count()
)

print(
    "Distinct Airline Keys:",
    traffic_valid
    .select("airline_key")
    .distinct()
    .count()
)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 11, Finished, Available, Finished, False)

Null Airline Keys: 0
Null Route Keys: 0
Distinct Airline Keys: 144


In [10]:
traffic_key_window = Window.orderBy(
    "date_key",
    "airline_key",
    "origin_airport_key",
    "destination_airport_key",
    "aircraft_type"
)

fact_airline_traffic_new = (
    traffic_valid
    .withColumn(
        "traffic_key",
        F.row_number().over(traffic_key_window)
    )
    .select(
        "traffic_key",
        "date_key",
        "airline_key",
        "origin_airport_key",
        "destination_airport_key",
        "route_key",
        "aircraft_type",
        "departures_scheduled",
        "payload",
        "seats",
        "passengers",
        "freight",
        "mail",
        "distance",
        "ramp_to_ramp",
        "air_time"
    )
)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 12, Finished, Available, Finished, False)

In [11]:
print("Final Traffic Rows:", fact_airline_traffic_new.count())

print(
    "Distinct Traffic Keys:",
    fact_airline_traffic_new
    .select("traffic_key")
    .distinct()
    .count()
)

print(
    "Null Airline Keys:",
    fact_airline_traffic_new
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Distinct Airline Keys:",
    fact_airline_traffic_new
    .select("airline_key")
    .distinct()
    .count()
)

fact_airline_traffic_new.printSchema()

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 13, Finished, Available, Finished, False)

Final Traffic Rows: 34902
Distinct Traffic Keys: 34902
Null Airline Keys: 0
Distinct Airline Keys: 144
root
 |-- traffic_key: integer (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- origin_airport_key: integer (nullable = true)
 |-- destination_airport_key: integer (nullable = true)
 |-- route_key: integer (nullable = true)
 |-- aircraft_type: long (nullable = true)
 |-- departures_scheduled: long (nullable = true)
 |-- payload: long (nullable = true)
 |-- seats: long (nullable = true)
 |-- passengers: long (nullable = true)
 |-- freight: long (nullable = true)
 |-- mail: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- ramp_to_ramp: long (nullable = true)
 |-- air_time: long (nullable = true)



In [12]:
traffic_key_validation.show(truncate=False)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 14, Finished, Available, Finished, False)

+--------------+-----------------+------------------------+-----------------------------+---------------+
|null_date_keys|null_airline_keys|null_origin_airport_keys|null_destination_airport_keys|null_route_keys|
+--------------+-----------------+------------------------+-----------------------------+---------------+
|0             |0                |731                     |710                          |0              |
+--------------+-----------------+------------------------+-----------------------------+---------------+



In [13]:
(
    fact_airline_traffic_new
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fact_airline_traffic")
)

print("fact_airline_traffic persisted successfully.")

persisted_traffic = spark.table("fact_airline_traffic")

print("Persisted Fact Rows:", persisted_traffic.count())

print(
    "Persisted Distinct Traffic Keys:",
    persisted_traffic
    .select("traffic_key")
    .distinct()
    .count()
)

print(
    "Persisted Null Airline Keys:",
    persisted_traffic
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Persisted Distinct Airline Keys:",
    persisted_traffic
    .select("airline_key")
    .distinct()
    .count()
)

StatementMeta(, 524a6f12-d2c3-4cec-b694-81e2b28f9a41, 15, Finished, Available, Finished, False)

fact_airline_traffic persisted successfully.
Persisted Fact Rows: 34902
Persisted Distinct Traffic Keys: 34902
Persisted Null Airline Keys: 0
Persisted Distinct Airline Keys: 144
